# Basic Lineage

AI-powered lineage explanation for a revenue pipeline.

Pipeline: `prices * quantities + bonus = total_revenue`

This notebook builds the pipeline, preserves all intermediate tables under `PreservationMode.FULL`, then uses an LLM to explain how the result was produced and to answer a debugging question via an agentic tool loop.

In [ ]:
!pip install 'aaiclick[ai]'
!python -m aaiclick setup

# Run Ollama inside Colab (adds ~2-4 min of cold-start on first run).
# llama3.1:8b fits a T4 GPU (~4.7 GB Q4) and gives much stronger
# tool-calling and reasoning than llama3.2:3b on the lineage tasks.
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > /tmp/ollama.log 2>&1 &
!sleep 3 && ollama pull llama3.1:8b

import logging
import os

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
)

os.environ.setdefault("AAICLICK_AI_MODEL", "ollama/llama3.1:8b")

# To use a hosted provider instead, replace the line above and set the key:
# LiteLLM model strings: https://docs.litellm.ai/docs/providers
#
#   os.environ["AAICLICK_AI_MODEL"] = "gemini/gemini-2.0-flash"
#   os.environ["GEMINI_API_KEY"] = "<your-key>"
#
#   os.environ["AAICLICK_AI_MODEL"] = "openai/gpt-4o-mini"
#   os.environ["OPENAI_API_KEY"] = "<your-key>"
#
#   os.environ["AAICLICK_AI_MODEL"] = "anthropic/claude-haiku-4-5-20251001"
#   os.environ["ANTHROPIC_API_KEY"] = "<your-key>"
#
# Colab users: load keys from the Secrets panel instead of pasting them here:
#   from google.colab import userdata
#   os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [ ]:
import asyncio

from aaiclick.ai.agents.debug_agent import debug_result
from aaiclick.ai.agents.lineage_agent import explain_lineage
from aaiclick.data.data_context import create_object_from_value
from aaiclick.data.object import Object
from aaiclick.oplog.lineage import OplogGraph, lineage_context, oplog_subgraph
from aaiclick.orchestration import (
    JobStatus,
    PreservationMode,
    ajob_test,
    get_tasks_for_job,
    job,
    task,
    tasks_list,
)
from aaiclick.orchestration.models import Task
from aaiclick.orchestration.orch_context import orch_context

## Pipeline tasks

In [ ]:
@task
async def create_prices() -> Object:
    return await create_object_from_value(
        [10.0, 20.0, 30.0, 40.0, 50.0],
        name="basic_lineage_prices",
    )


@task
async def create_quantities() -> Object:
    return await create_object_from_value(
        [2.0, 3.0, 1.0, 5.0, 4.0],
        name="basic_lineage_quantities",
    )


@task
async def compute_revenue(prices: Object, quantities: Object) -> Object:
    return await (prices * quantities)


@task
async def add_bonus(revenue: Object) -> Object:
    bonus = await create_object_from_value([5.0, 5.0, 5.0, 5.0, 5.0])
    return await (revenue + bonus)

## Job definition

In [ ]:
@job("revenue_pipeline")
def revenue_pipeline():
    prices = create_prices()
    quantities = create_quantities()
    revenue = compute_revenue(prices=prices, quantities=quantities)
    total = add_bonus(revenue=revenue)
    return tasks_list(prices, quantities, revenue, total)

## Report rendering

In [ ]:
def _print_graph(
    graph: OplogGraph,
    title: str,
    root: str,
    labels: dict[str, str],
) -> None:
    print(f"\n## {title}\n")
    print(f"- Root: `{labels.get(root, root)}`")
    print(f"- Operations: {len(graph.nodes)}")
    print(f"- Edges: {len(graph.edges)}\n")
    for edge in graph.edges:
        src = labels.get(edge.source, edge.source)
        tgt = labels.get(edge.target, edge.target)
        print(f"- `{src}` -> `{tgt}` (via `{edge.operation}`)")


def print_report(
    *,
    tasks: list[Task],
    target_table: str,
    backward_graph: OplogGraph,
    forward_graph: OplogGraph,
    source_table: str,
    explanation: str,
    debug_answer: str,
) -> None:
    backward_labels = backward_graph.build_labels()
    forward_labels = forward_graph.build_labels()

    print("## Pipeline Tasks\n")
    for t in tasks:
        if t.result:
            table = t.result.get("table", "")
            label = backward_labels.get(table, "")
            suffix = f" ({label})" if label else ""
            print(f"- **{t.name}**: {t.status.value}{suffix}")
        else:
            print(f"- **{t.name}**: {t.status.value}")

    _print_graph(backward_graph, "Backward Lineage Graph", target_table, backward_labels)
    _print_graph(forward_graph, "Forward Lineage Graph", source_table, forward_labels)

    print("\n## AI Explanation (backward lineage)\n")
    print("**Question**: How was this table produced? What arithmetic was applied?\n")
    print(explanation)

    print("\n## AI Debug (agentic tool-calling)\n")
    print("**Question**: Which row has the highest value and which inputs drove it?\n")
    print(debug_answer)

## Run the pipeline

In [ ]:
async def main():
    async with orch_context():
        pipeline = await revenue_pipeline(
            preservation_mode=PreservationMode.FULL,
        )
        await ajob_test(pipeline)
        assert pipeline.status == JobStatus.COMPLETED, f"Job failed: {pipeline.error}"

        tasks = await get_tasks_for_job(pipeline.id)
        target_table = next(t for t in tasks if t.name == "add_bonus").result["table"]
        source_table = next(t for t in tasks if t.name == "create_prices").result["table"]

        async with lineage_context():
            backward_graph, forward_graph = await asyncio.gather(
                oplog_subgraph(target_table, direction="backward"),
                oplog_subgraph(source_table, direction="forward"),
            )
            explanation = await explain_lineage(
                target_table,
                question="How was this table produced? What arithmetic was applied?",
                graph=backward_graph,
            )
            debug_answer = await debug_result(
                target_table,
                question=(
                    "Which output row has the highest value, and which input "
                    "rows drove it? Use the tools to inspect the tables."
                ),
                graph=backward_graph,
            )

        print_report(
            tasks=tasks,
            target_table=target_table,
            backward_graph=backward_graph,
            forward_graph=forward_graph,
            source_table=source_table,
            explanation=explanation,
            debug_answer=debug_answer,
        )

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] [%(levelname)s] [%(name)s] %(message)s",
    force=True,
)

await main()